# Matching Engine (CV vs Job Offer)
## Imports

In [9]:
from pydantic import BaseModel
from openai import OpenAI
import pymupdf
from sentence_transformers import SentenceTransformer, util
import torch
import json 

## Initialization

In [10]:
# 1. Load the NLP mathematical model
nlp_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Connect to Ollama (must be running in the background)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" 
)

# 3. Define the expected JSON format
class SkillsExtraction(BaseModel):
    found_skills: list[str]
    required_skills: list[str]

print("✅ Initialization complete. Engines ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4203.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Initialization complete. Engines ready.


## Input Data

In [11]:
cv_pdf_path = "../data/cv_sample_cleaned/moubarak_ezzyani.pdf" 

job_description = """
Mastery of Python 3
Experience with SQL databases (PostgreSQL, SQLite)
Experience with Git, Docker, and CI/CD pipelines
Notions in Machine Learning (Pandas, Scikit-learn)
Team spirit
Good communication skills
"""

# Read the PDF
doc = pymupdf.open(cv_pdf_path)
cv_text = ""
for page in doc:
    cv_text += page.get_text()
    
print(f"✅ CV read successfully. Length: {len(cv_text)} characters.")

✅ CV read successfully. Length: 1830 characters.


## Smart Extraction (Qwen 2.5)

In [12]:
print("⏳ Qwen is reading and extracting skills...")

system_prompt = """
You are an HR expert. Your only mission is to extract technical and soft skill keywords.
1. Find all the skills present in the provided CV text.
2. Find all the skills required in the provided job description text.
Do not write any sentences. Only fill out the requested JSON format.
"""

user_prompt = f"--- CV TEXT ---\n{cv_text}\n\n--- JOB DESCRIPTION ---\n{job_description}"

response = client.beta.chat.completions.parse(
    model="qwen2.5:3b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format=SkillsExtraction, 
)

extracted_data = response.choices[0].message.parsed
cv_skills = extracted_data.found_skills
job_skills = extracted_data.required_skills

print("✅ Skills extracted by Qwen!")
print(f"-> {len(cv_skills)} skills found in the CV.")
print(f"-> {len(job_skills)} skills required by the job offer.")

⏳ Qwen is reading and extracting skills...
✅ Skills extracted by Qwen!
-> 37 skills found in the CV.
-> 6 skills required by the job offer.


## Math & Scoring

In [13]:
cv_embeddings = nlp_model.encode(cv_skills, convert_to_tensor=True)
job_embeddings = nlp_model.encode(job_skills, convert_to_tensor=True)

total_score = 0
missing_skills = []

# Cosine similarity matrix calculation
cosine_scores = util.cos_sim(job_embeddings, cv_embeddings)

for i, job_skill in enumerate(job_skills):
    best_match_score = torch.max(cosine_scores[i]).item()
    total_score += best_match_score
    
    # If the similarity is < 40%, it is considered a missing skill
    if best_match_score < 0.4:
        missing_skills.append(job_skill)

final_match_score = round((total_score / len(job_skills)) * 100, 2)
print(f"✅ Calculation complete. Score: {final_match_score}%")

✅ Calculation complete. Score: 49.81%


## Display Results

In [14]:
final_result = {
    "status": "success",
    "match_score": final_match_score,
    "data": {
        "cv_skills_found": cv_skills,
        "job_skills_required": job_skills,
        "missing_skills": missing_skills
    }
}

# formatted JSON & indentation
print(json.dumps(final_result, indent=4, ensure_ascii=False))

{
    "status": "success",
    "match_score": 49.81,
    "data": {
        "cv_skills_found": [
            "Python",
            "TensorFlow",
            "PyTorch",
            "Scikit-learn",
            "OpenCV",
            "RAG",
            "Embeddings",
            "Vector DB",
            "FastAPI",
            "Next.js",
            "TypeScript",
            "APIs REST",
            "Agile",
            "Scrum",
            "UML",
            "Docker",
            "Kubernetes",
            "Airflow",
            "MLflow",
            "GitHub Actions",
            "Linux",
            "Git",
            "SQL (MySQL, PostgreSQL, Oracle)",
            "FastAPI (Python)",
            "Java",
            "C++",
            "Bash/Shell",
            "SQL",
            "English (Advanced Technical)",
            "French (Current)",
            "Arabic (Native)",
            "Hugging Face API",
            "JWT",
            "Machine Learning",
            "Pandas",
            "Git"

## load data

In [15]:
# # --- Define the path to test PDF file
# cv_path = "../data/cv_sample_cleaned/moubarak_ezzyani.pdf" 

# # extract all text from a PDF
# def extract_text_from_pdf(file_path):
#     full_text = ""
#     try:
#         with pdfplumber.open(file_path) as pdf:
#             for page in pdf.pages:
#                 full_text += page.extract_text() + "\n"
#         return full_text
#     except Exception as e:
#         return f"Error reading the PDF: {e}"

# # --- CV into a variable
# raw_cv_text = extract_text_from_pdf(cv_path)

# # --- Display the results 
# print("======================================================")
# print("📄 EXTRACTED CV PREVIEW (First 500 characters):")
# print("======================================================")
# print(raw_cv_text[:500] + "...\n") # Display only the beginning to avoid cluttering the screen



### fake opportunity

In [16]:
# # --- fake Job Description for test
# job_description_text = """
# We are looking for a Python Backend Developer (M/F).
# Required skills:
# - Mastery of Python 3 and the FastAPI or Django framework.
# - Good knowledge of SQL databases (PostgreSQL, SQLite).
# - Experience with Git, Docker, and CI/CD pipelines.
# - Notions in Machine Learning (Pandas, Scikit-learn) are a plus.
# The candidate must show team spirit and good communication skills.
# """


# print("======================================================")
# print("🎯 TARGET JOB DESCRIPTION:")
# print("======================================================")
# print(job_description_text)